In [0]:
%python
import sys
sys.path.append("/Workspace/Users/alliesdl@gmail.com/MLops_bal_pipeline")   # your Git folder
from pyspark.sql import functions as F
from src.quality import assert_unique

In [0]:
%python
meta = spark.table("clinical_portfolio.silver.metadata").select(
    "pt_id", "sex", "age_years", "bmi", "smoking_status", "drug_use", "vaping",
    "hiv_status", "cd4_count", "viral_load_cp_ml", "art", "genexpert_res",
    "genexpert_rif", "smear_grade", "cxr_finding", "igra_result", "igra_value_iu_ml")

bal0 = (spark.table("clinical_portfolio.silver.bal").filter("visit = 'Baseline'")
        .select("pt_id", "viability_pct", "recovery_ratio", "total_cells",
                "macrophages_pct", "lymphocytes_pct", "neutrophils_pct",
                "time_to_process_min"))

trt = spark.table("clinical_portfolio.silver.treatment").select("pt_id", "outcome_group")

gold = (meta.join(bal0, "pt_id", "left")      # keep everyone, even without a BAL row
            .join(trt, "pt_id", "inner")      # keep only people with an outcome
            .filter(F.col("outcome_group").isNotNull())
            .withColumn("label", (F.col("outcome_group") == "favourable").cast("int")))

assert_unique(gold, "pt_id")                   # a bad join would multiply rows
print(gold.count(), gold.groupBy("label").count().collect())
(gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
     .saveAsTable("clinical_portfolio.gold.model_table"))